# 02. Data Cleaning & Outlier Analysis

## Objectives:
1. Verify schema consistency and non-null constraints.
2. Detect and handle exact duplicate transaction records.
3. Conduct rigorous statistical outlier investigation on `Amount`.
4. Explain why outliers must NOT be dropped blindly in fraud detection.
5. Save sanitized data to `dataset/processed/`.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
from src.config import RAW_DATA_PATH, CLEANED_DATA_PATH
from src.data_loader import load_raw_data
from src.data_cleaning import validate_schema, clean_missing_and_inf, handle_duplicates, analyze_outliers

df = load_raw_data(RAW_DATA_PATH)
validate_schema(df)


### Handling Missing & Infinite Values


In [ ]:
df_clean = clean_missing_and_inf(df)
print('Total missing cells after check:', df_clean.isnull().sum().sum())


### Duplicate Record Detection

Exact duplicate records often occur from payment network retry bursts or dual-logging.


In [ ]:
df_dedup, dup_count = handle_duplicates(df_clean, drop=True)
print(f'Dropped {dup_count:,} duplicate transactions. Remaining clean rows: {len(df_dedup):,}')


### Outlier Analysis: Why We Retain Financial Outliers

In standard regression, extreme outliers (> Q3 + 1.5*IQR) are frequently truncated. In financial fraud analytics, this causes severe survivorship bias because high-value transactions harbor critical fraud cases.


In [ ]:
outlier_analysis = analyze_outliers(df_dedup)
for k, v in outlier_analysis.items():
    print(f'{k}: {v}')


### Persisting the Cleaned Dataset


In [ ]:
CLEANED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.to_csv(CLEANED_DATA_PATH, index=False)
print(f'Cleaned dataset successfully saved to: {CLEANED_DATA_PATH}')
